# IMDb Top Netflix Movies and TV Shows

Dataset link: https://www.kaggle.com/datasets/bharatnatrayn/movies-dataset-for-feature-extracion-prediction?select=movies.csv

## Contents
- [Imports and Loading](#imports-and-loading)
- [Overview](#overview)
  - [Whitespace and Column Cleaning](#whitespace-and-column-cleaning)
  - [Duplicate Handling](#duplicate-handling)
  - [Overall Insights](#overall-insights)
  - [Rejected records creation](#rejected-records-creation)
  - [Before and After records](#before-and-after-records)
- [Data Fromatting and Type Conversion](#data-fromatting-and-type-conversion)
  - Numerical data conversion
  - [Formatting strings](#formatting-strings)
  - [Conclusion](#conclusion)
- [ROMI Approach](#romi-approach)
  - [Relation](#relation)
  - [Outlier](#outlier)
    - Investigating `Column`
  - [Mismatch](#mismatch)
  - [Interpolation](#interpolation)
- [Final Check](#final-check)
- [Before VS After](#before-vs-after)
- [Export](#export)

## Imports and Loading

In [1]:
import pandas as pd
import re

In [2]:
df = pd.read_csv("data/raw.csv")

print(f"Loaded dataset: {df.shape[0]} rows, {df.shape[1]} columns")

Loaded dataset: 9999 rows, 9 columns


## Overview

In [3]:
display(df.head())

,MOVIES,YEAR,GENRE,RATING,ONE-LINE,STARS,VOTES,RunTime,Gross
0,Blood Red Sky,(2021),"\nAction, Horror, Thriller",6.1,\nA woman with a mysterious illness is forced ...,\n Director:\nPeter Thorwarth\n| \n Star...,"21,062",121.0,NaN
1,Masters of the Universe: Revelation,(2021– ),"\nAnimation, Action, Adventure",5.0,\nThe war for Eternia begins again in what may...,"\n \n Stars:\nChris Wood, \nSara...","17,870",25.0,NaN
2,The Walking Dead,(2010–2022),"\nDrama, Horror, Thriller",8.2,\nSheriff Deputy Rick Grimes wakes up from a c...,"\n \n Stars:\nAndrew Lincoln, \n...","885,805",44.0,NaN
3,Rick and Morty,(2013– ),"\nAnimation, Adventure, Comedy",9.2,\nAn animated series that follows the exploits...,"\n \n Stars:\nJustin Roiland, \n...","414,849",23.0,NaN
4,Army of Thieves,(2021),"\nAction, Crime, Horror",NaN,"\nA prequel, set before the events of Army of ...",\n Director:\nMatthias Schweighöfer\n| \n ...,NaN,NaN,NaN


### Whitespace and Column Cleaning

In [4]:
# whitespace cleaning
cols = df.select_dtypes(include=["object", "str", "string"]).columns.tolist()

for col in cols:
    df[col] = df[col].str.replace(r"[\t\n]+", " ", regex=True).str.replace(r"\s{2,}", " ", regex=True).str.strip()

In [5]:
# column cleaning
print("Initial columns:", df.columns.tolist())
df.columns = df.columns.str.strip().str.lower().str.replace("-", "_")
print("Clean columns:", df.columns.tolist())

Initial columns: ['MOVIES', 'YEAR', 'GENRE', 'RATING', 'ONE-LINE', 'STARS', 'VOTES', 'RunTime', 'Gross']
Clean columns: ['movies', 'year', 'genre', 'rating', 'one_line', 'stars', 'votes', 'runtime', 'gross']


### Duplicate Handling

In [6]:
print("(1) Duplicates:", df.duplicated(subset=["movies", "one_line"]).sum())
display(df.loc[df.duplicated(subset=["movies", "one_line"], keep=False)].sort_values(by="movies").head(10))

(1) Duplicates: 857


,movies,year,genre,rating,one_line,stars,votes,runtime,gross
9992,1899,(2022– ),"Drama, History, Horror",NaN,Add a Plot,Director: Baran bo Odar,NaN,NaN,NaN
9989,1899,(2022– ),"Drama, History, Horror",NaN,Add a Plot,Director: Baran bo Odar,NaN,NaN,NaN
9990,1899,(2022– ),"Drama, History, Horror",NaN,Add a Plot,Director: Baran bo Odar,NaN,NaN,NaN
9443,1899,(2022– ),"Drama, History, Horror",NaN,Add a Plot,Director: Baran bo Odar | Stars: Aneurin Barna...,NaN,NaN,NaN
9991,1899,(2022– ),"Drama, History, Horror",NaN,Add a Plot,Director: Baran bo Odar,NaN,NaN,NaN
9982,1899,(2022– ),"Drama, History, Horror",NaN,Add a Plot,Director: Baran bo Odar,NaN,NaN,NaN
9980,1899,(2022– ),"Drama, History, Horror",NaN,Add a Plot,Director: Baran bo Odar,NaN,NaN,NaN
9981,1899,(2022– ),"Drama, History, Horror",NaN,Add a Plot,Director: Baran bo Odar,NaN,NaN,NaN
9177,800 metros,(2021– ),Documentary,NaN,Add a Plot,Director: León Siminiani,NaN,NaN,NaN
9178,800 metros,(2021– ),Documentary,NaN,Add a Plot,Director: León Siminiani,NaN,NaN,NaN


*Question 1 for the client*

---

Hi,

I noticed the dataset doesn’t have a unique ID column. This makes it difficult to confidently identify duplicates.

In addition, there are no indications of what type the record is (Movie/Series). And if series, no season and episode number. This makes it even more difficult to say if two identical records are duplicates or different episodes of the same TV series. Hence, **I will proceed by assuming there are no duplicate rows and treating identical records distinctively**. Though, I’ll make sure everything is document clearly.

Is this approach fine with you?

### Overall Insights

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9999 entries, 0 to 9998
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   movies    9999 non-null   str    
 1   year      9355 non-null   str    
 2   genre     9919 non-null   str    
 3   rating    8179 non-null   float64
 4   one_line  9999 non-null   str    
 5   stars     9999 non-null   str    
 6   votes     8179 non-null   str    
 7   runtime   7041 non-null   float64
 8   gross     460 non-null    str    
dtypes: float64(2), str(7)
memory usage: 703.2 KB


In [8]:
df.describe()

,rating,runtime
count,8179.000000,7041.000000
mean,6.921176,68.688539
std,1.220232,47.258056
min,1.100000,1.000000
25%,6.200000,36.000000
50%,7.100000,60.000000
75%,7.800000,95.000000
max,9.900000,853.000000


In [9]:
df.isna().sum()

movies         0
year         644
genre         80
rating      1820
one_line       0
stars          0
votes       1820
runtime     2958
gross       9539
dtype: int64

## Data Formatting and Type Conversion

### Formatting Strings

In [10]:
str_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()

print("String type columns:", str_cols)
display(df[str_cols].head(3))

String type columns: ['movies', 'year', 'genre', 'one_line', 'stars', 'votes', 'gross']


,movies,year,genre,one_line,stars,votes,gross
0,Blood Red Sky,(2021),"Action, Horror, Thriller",A woman with a mysterious illness is forced in...,Director: Peter Thorwarth | Stars: Peri Baumei...,"21,062",NaN
1,Masters of the Universe: Revelation,(2021– ),"Animation, Action, Adventure",The war for Eternia begins again in what may b...,"Stars: Chris Wood, Sarah Michelle Gellar, Lena...","17,870",NaN
2,The Walking Dead,(2010–2022),"Drama, Horror, Thriller",Sheriff Deputy Rick Grimes wakes up from a com...,"Stars: Andrew Lincoln, Norman Reedus, Melissa ...","885,805",NaN


The hyphens in entries of the year column are not the ordinary hyphens -, but instead an **En Dash** or **Em Dash** (UNICODE **\u2013** and **\u2014** respectively). The beneath cell replaces these unusual dashes with the common hyphen -.

In [11]:
df["year"] = df.year.str.replace("\u2013", "-").str.replace("\u2014", "-")

Now we proceed to correcting formats.

In [12]:
def format_masker(df: pd.DataFrame, col_format: dict, show=False):
    for col, format in col_format.items():
        mask = df[col].str.match(format)

        print(f"Non-matching values for column {col} and format {format}:", (~mask).sum())
        if show:
            display(df[col][(~mask)])
            print("-"*64)

col_format_pair = {
    "year": r"\(\d{4}\-?\s?(?:\d{4})?\)",
    }

format_masker(df, col_format_pair, show=True)

Non-matching values for column year and format \(\d{4}\-?\s?(?:\d{4})?\): 1692


36           (I) (2018- )
84          (II) (2007- )
125            (I) (2019)
155     (2021 TV Special)
161            (I) (2017)
              ...        
9909                  NaN
9910                  NaN
9911                  NaN
9912                  NaN
9913                  NaN
Name: year, Length: 1692, dtype: str

----------------------------------------------------------------


In [13]:
capture_pattern = r"(.*)(\(\d{4}\-?\s?(?:\d{4})?\))(.*)"

df["year"] = df.year \
    .str.replace(r"[a-zA-Z\s]+", "", regex=True) \
    .str.replace(capture_pattern, r"\2", regex=True)
    # remove letters and spaces anywhere in the text
    # replace the entire string with only the desired capture group


In [14]:
year_pattern = col_format_pair["year"]

year_mask = df.year.str.match(year_pattern, na=False) # na=False so existing nulls count as non-matching

df.loc[~year_mask, "year"] = pd.NA

year_matching_count = df[year_mask].year.shape[0]
year_non_matching_count = df[~year_mask].year.shape[0]
print("Matching values:", year_matching_count)
print("Non-matching values:", year_non_matching_count)
print("Total:", year_matching_count + year_non_matching_count)

Matching values: 9251
Non-matching values: 748
Total: 9999


The only column I checked the formatting is the year column. This is because the other columns are filled with a bunch of texts (no proper format or pattern / unstructured). I may check for extreme absurdity in each column but I prefer to assume there is no such a thing.

As a result, after formatting the year column, the number of NA values has increased from 644 to 748. Right now, it is guaranteed every valid data inside follows the format `\(\d{4}\-?\s?(?:\d{4})?\)`.

### `gross` to Float64

In [15]:
valid_gross_values = df.gross.dropna().sort_values()

display(valid_gross_values)

1111     $0.00M
4637     $0.00M
4641     $0.00M
4648     $0.00M
4783     $0.00M
         ...   
793     $94.90M
383     $95.86M
262     $96.52M
582     $97.10M
421     $97.69M
Name: gross, Length: 460, dtype: str

Seems like every valid value follows the format `\$\d+\.\d{2}M`.

In [16]:
gross_mask = df.gross.str.fullmatch(r"\$\d+\.\d{2}M")

display(df.gross[gross_mask])

print(r"Every valid gross value follow the format \$\d+\.\d{2}M >>>", gross_mask.sum() == valid_gross_values.shape[0])

77       $75.47M
85      $402.45M
95       $89.22M
111     $315.54M
125      $57.01M
          ...   
5750      $0.09M
5770      $0.00M
5835      $0.01M
6056      $0.01M
6292      $0.14M
Name: gross, Length: 460, dtype: str

Every valid gross value follow the format \$\d+\.\d{2}M >>> True


In [17]:
def parse_gross(gross_str: str):
    if pd.isna(gross_str):
        return pd.NA
    
    s = gross_str.replace("$", "").replace("M", "")

    return float(s) * 1e6

df.gross = df.gross.apply(parse_gross).astype("Float64")

In [18]:
display(df.gross.dropna())

77       75470000.0
85      402450000.0
95       89220000.0
111     315540000.0
125      57010000.0
           ...     
5750        90000.0
5770            0.0
5835        10000.0
6056        10000.0
6292       140000.0
Name: gross, Length: 460, dtype: Float64

### `votes` to Int64

In [26]:
df["votes"] = df.votes.str.replace(",", "").astype("Int64")

In [27]:
df.votes.describe()

count          8179.0
mean     15124.062722
std        70054.5777
min               5.0
25%             166.0
50%             789.0
75%            3772.0
max         1713028.0
Name: votes, dtype: Float64